# Deep Learning: FC, CNN, RNN, GAN, Autoencoders
**Author:** Kseniia Tikhonova  
**Description:** Comparative study of deep learning architectures across four domains: synthetic data, computer vision, sequential modelling, and generative models.

## Section 1: Reproducibility — Seeds & Device Config
All seeds are set before any data loading or model initialisation. The remaining source of nondeterminism is GPU kernel non-determinism in cuDNN backward passes (e.g. Conv2D). Setting `TF_DETERMINISTIC_OPS=1` reduces but does not fully eliminate this on all hardware/driver combinations.

In [1]:
import os, re, csv, random, struct, unittest
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
import nltk
from datetime import datetime
from skimage import io, transform
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras import layers, Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense, Flatten, Dropout, Input,
    Conv2D, MaxPooling2D, UpSampling2D, Conv2DTranspose,
    LSTM, Embedding, BatchNormalization, LeakyReLU,
    GlobalAveragePooling2D, Reshape,
    RandomFlip, RandomRotation, RandomZoom, RandomContrast
)
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, CSVLogger
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.losses import BinaryCrossentropy, MeanSquaredError
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.applications import ResNet50, VGG16
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from nltk.stem import WordNetLemmatizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.19.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Section 2: Experiment Logger
Tracks every run (name, params, metrics, timestamp) into a single `experiment_log.csv`. This was the primary tool used to compare architectures and inform decisions — e.g. which ResNet variant to proceed with.

In [2]:
LOG_FILE = "experiment_log.csv"

def log_experiment(name, params, metrics):
    row = {"timestamp": datetime.now().isoformat(), "experiment": name}
    row.update(params)
    row.update(metrics)
    file_exists = os.path.isfile(LOG_FILE)
    with open(LOG_FILE, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=row.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)
    print(f"[LOG] {name} -> {metrics}")